# 7CS353 Machine Learning Lab — Assignment 7
### Classification: Logistic Regression and Perceptron Learning

Name: Atharv V Kulkarni
PRN: 245109074
Class: TY CSE(B)

## B1. Load and explore

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
np.random.seed(42)

In [ ]:
import os
if os.path.exists("student_pass.csv"):
    df = pd.read_csv("student_pass.csv")
else:
    rng = np.random.default_rng(seed=42)
    n = 500
    study = rng.normal(5, 1.8, n).clip(0.5, 10).round(2)
    attendance = rng.normal(72, 15, n).clip(30, 100).round(1)
    z = -5 + 0.6*study + 0.04*attendance
    prob = 1 / (1 + np.exp(-z))
    y = rng.binomial(1, prob, n)
    df = pd.DataFrame({"study_hours": study, "attendance": attendance, "pass": y})
    df.to_csv("student_pass.csv", index=False)
print(df.shape)
print(df["pass"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
colors = df["pass"].map({0: "crimson", 1: "royalblue"})
ax.scatter(df["study_hours"], df["attendance"], c=colors, edgecolor="k", alpha=0.75)
ax.set_xlabel("study hours")
ax.set_ylabel("attendance")
ax.set_title("study hours vs attendance by pass/fail")
plt.show()

**Solution:**

_(fill in after viewing the scatter plot — comment on whether the red and blue points form two roughly separable regions or whether they overlap heavily in the middle of the feature space)_

## B2. Preprocessing

In [ ]:
x = df[["study_hours", "attendance"]].values
y = df["pass"].values
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)
print(x_train_s.mean(axis=0), x_train_s.std(axis=0))

## B3. Logistic regression from scratch

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_loss(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))

In [ ]:
def train_logistic(x, y, lr=0.1, epochs=2000):
    n, d = x.shape
    w = np.zeros(d)
    b = 0.0
    losses = []
    for epoch in range(epochs):
        z = x @ w + b
        y_hat = sigmoid(z)
        loss = bce_loss(y, y_hat)
        losses.append(loss)
        dw = (1/n) * x.T @ (y_hat - y)
        db = (1/n) * np.sum(y_hat - y)
        w -= lr * dw
        b -= lr * db
    return w, b, losses

w_log, b_log, loss_history = train_logistic(x_train_s, y_train)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(loss_history, color="darkorange")
ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy loss")
ax.set_title("logistic regression training loss")
plt.show()

In [ ]:
print("scratch w:", w_log, "scratch b:", b_log)
sk_log = LogisticRegression()
sk_log.fit(x_train_s, y_train)
print("sklearn w:", sk_log.coef_[0], "sklearn b:", sk_log.intercept_[0])

**Solution:**

_(fill in after running — compare the scratch weights/bias with the sklearn LogisticRegression coefficients and note how close they are)_

## B4. Perceptron from scratch

In [ ]:
def step(z):
    return np.where(z >= 0, 1, 0)

def train_perceptron(x, y, lr=0.1, epochs=100):
    n, d = x.shape
    w = np.zeros(d)
    b = 0.0
    mistakes_per_epoch = []
    for epoch in range(epochs):
        mistakes = 0
        for i in range(n):
            z = np.dot(w, x[i]) + b
            y_hat = step(z)
            if y_hat != y[i]:
                w = w + lr*(y[i]-y_hat)*x[i]
                b = b + lr*(y[i]-y_hat)
                mistakes += 1
        mistakes_per_epoch.append(mistakes)
    return w, b, mistakes_per_epoch

w_perc, b_perc, mistake_history = train_perceptron(x_train_s, y_train)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(mistake_history, color="seagreen")
ax.set_xlabel("epoch")
ax.set_ylabel("misclassifications")
ax.set_title("perceptron misclassifications per epoch")
plt.show()

In [ ]:
print("scratch w:", w_perc, "scratch b:", b_perc)
sk_perc = Perceptron()
sk_perc.fit(x_train_s, y_train)
print("sklearn w:", sk_perc.coef_[0], "sklearn b:", sk_perc.intercept_[0])

**Solution:**

_(fill in after running — compare the scratch perceptron weights/bias with the sklearn Perceptron, and note whether the misclassification curve reaches zero)_

## B5. Evaluation

In [ ]:
z_log_test = x_test_s @ w_log + b_log
y_prob_log = sigmoid(z_log_test)
y_pred_log = (y_prob_log >= 0.5).astype(int)

z_perc_test = x_test_s @ w_perc + b_perc
y_pred_perc = step(z_perc_test)

In [ ]:
cm_log = confusion_matrix(y_test, y_pred_log)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm_log, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("logistic regression confusion matrix")
plt.show()

In [ ]:
cm_perc = confusion_matrix(y_test, y_pred_perc)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm_perc, annot=True, fmt="d", cmap="Greens", ax=ax)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("perceptron confusion matrix")
plt.show()

In [ ]:
print("logistic regression report")
print(classification_report(y_test, y_pred_log))
print("perceptron report")
print(classification_report(y_test, y_pred_perc))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob_log)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, color="crimson", label="roc curve (auc = %.3f)" % roc_auc)
ax.plot([0,1],[0,1], linestyle="--", color="gray")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("roc curve — logistic regression")
ax.legend()
plt.show()

In [ ]:
acc_log = accuracy_score(y_test, y_pred_log)
prec_log = precision_score(y_test, y_pred_log)
rec_log = recall_score(y_test, y_pred_log)
f1_log = f1_score(y_test, y_pred_log)

acc_perc = accuracy_score(y_test, y_pred_perc)
prec_perc = precision_score(y_test, y_pred_perc)
rec_perc = recall_score(y_test, y_pred_perc)
f1_perc = f1_score(y_test, y_pred_perc)

print("logistic regression", acc_log, prec_log, rec_log, f1_log, roc_auc)
print("perceptron", acc_perc, prec_perc, rec_perc, f1_perc)

| Model | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| Logistic regression | _ | _ | _ | _ | _ |
| Perceptron | _ | _ | _ | _ | N/A |

_(fill in the blanks with the printed values from the cell above)_

**Solution:**

_(fill in after running — state which model scored higher on each metric and why that might be expected)_

## B6. Decision boundaries

In [ ]:
def plot_boundary(ax, w, b, scaler, predict_fn, title):
    x_min, x_max = df["study_hours"].min()-0.5, df["study_hours"].max()+0.5
    y_min, y_max = df["attendance"].min()-2, df["attendance"].max()+2
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_s = scaler.transform(grid)
    zz = predict_fn(grid_s, w, b).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5,0.5,1.5], colors=["mistyrose","lightsteelblue"], alpha=0.6)
    colors = df["pass"].map({0: "crimson", 1: "royalblue"})
    ax.scatter(df["study_hours"], df["attendance"], c=colors, edgecolor="k", alpha=0.8)
    ax.set_xlabel("study hours")
    ax.set_ylabel("attendance")
    ax.set_title(title)

def predict_log(x_s, w, b):
    return (sigmoid(x_s @ w + b) >= 0.5).astype(int)

def predict_perc(x_s, w, b):
    return step(x_s @ w + b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
plot_boundary(axes[0], w_log, b_log, scaler, predict_log, "logistic regression decision boundary")
plot_boundary(axes[1], w_perc, b_perc, scaler, predict_perc, "perceptron decision boundary")
plt.tight_layout()
plt.show()

**Solution:**

_(fill in after viewing the two boundaries — describe any region where they differ and explain why, referring to the convex-loss vs error-driven update difference from Part A)_

## B7. Interpretation

**Solution:**

_(write a 5–7 sentence paragraph after completing B1–B6, addressing: which model performed better and why; what the logistic regression coefficients reveal about the relative importance of study hours vs attendance; why the perceptron cannot produce an AUC score; and in what scenario you would prefer logistic regression over the perceptron)_